# Qwen Image Edit GGUF Web Server (ComfyUI) — Google colab

ComfyUI APIをバックエンドとして、Web サーバーを起動し、cloudflared で外部公開します。

**必要環境:**
 - GPUが使えるランタイム(A100)

## 1. torchのインストール

In [ ]:
!pip install -U huggingface_hub requests urllib3 charset-normalizer

import os
import sys
import subprocess
import platform

def get_system_cuda_version():
    """nvidia-smiを使用してシステムのCUDAバージョンを取得する"""
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits"]).decode()
        # CUDAバージョンを直接取得する（もしくはnvccから取得）
        cuda_out = subprocess.check_output(["nvcc", "--version"]).decode()
        for line in cuda_out.split('\n'):
            if "release" in line:
                return line.split("release ")[1].split(",")[0].replace(".", "")
    except:
        # 万が一取得失敗した場合はデフォルトを指定（例: 130）
        return "130"
    return "130"

def run_command(command):
    print(f"Executing: {command}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + command.split())

# 1. CUDAバージョンの取得
cuda_tag = f"cu{get_system_cuda_version()}"
print(f"🚀 Detected System CUDA: {cuda_tag}")

# 2. PyTorchのインストール (CUDAバージョンに合わせる)
print(f"📦 Installing PyTorch for {cuda_tag}...")
torch_index_url = f"https://download.pytorch.org/whl/{cuda_tag}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch", "torchvision", "--index-url", torch_index_url])

print("\nInstallations completed successfully!")

## 2. パッケージインストール

In [ ]:
# ライブラリインストール
!apt -y remove python3-blinker
!pip install "flask>=3.0"  pillow psutil accelerate googletrans

# cloudflared インストール（）
!wget -qN https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# ComfyUI用ライブラリインストール
!pip install comfyui-frontend-package soundfile gguf tokenizer sentencepiece protobuf --extra-index-url $torch_index_url

# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!git pull

# 追加インストール
!pip install -r requirements.txt --extra-index-url $torch_index_url

%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/city96/ComfyUI-GGUF.git

%cd ../../

print("OK")

## 3. HuggingFace ログイン（任意）

ログインするとモデルダウンロードが高速化されます（レート制限緩和）。  
`HF_TOKEN` を登録しておくと自動認証されます。 
スキップも可。

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN") or getpass("Enter HF token: ")
    from huggingface_hub import login
    login(token=hf_token)
    print("HuggingFace: Colab Secrets で認証しました")
except Exception:
    print("Colab Secrets に HF_TOKEN が未登録です。手動ログインを試みます...")
    print("スキップする場合はこのセルの出力を無視してください。")
    try:
        from huggingface_hub import login
        login()
    except Exception:
        print("HuggingFace: 未ログイン（匿名ダウンロード）")

## 4. モデルのダウンロード

In [ ]:
!hf download Arunk25/Qwen-Image-Edit-Rapid-AIO-GGUF --local-dir ./hf_download --include "v23/Qwen-Rapid-NSFW-v23_Q6_K.gguf"
%mv ./hf_download/v23/Qwen-Rapid-NSFW-v23_Q6_K.gguf ./ComfyUI/models/diffusion_models/
!hf download mradermacher/Qwen2.5-VL-7B-Instruct-heretic-GGUF --local-dir ./hf_download --include "Qwen2.5-VL-7B-Instruct-heretic.Q6_K.gguf"
%mv ./hf_download/Qwen2.5-VL-7B-Instruct-heretic.Q6_K.gguf ./ComfyUI/models/text_encoders/
!hf download mradermacher/Qwen2.5-VL-7B-Instruct-heretic-GGUF --local-dir ./hf_download --include "Qwen2.5-VL-7B-Instruct-heretic.mmproj-Q8_0.gguf"
%mv ./hf_download/Qwen2.5-VL-7B-Instruct-heretic.mmproj-Q8_0.gguf ./ComfyUI/models/text_encoders/
!hf download Comfy-Org/Qwen-Image_ComfyUI --local-dir ./hf_download --include "split_files/vae/qwen_image_vae.safetensors"
%mv ./hf_download/split_files/vae/qwen_image_vae.safetensors ./ComfyUI/models/vae/

print("OK")

## 5. app_comfyui.py を配置

In [ ]:
import shutil

!git clone https://github.com/id-fa/simple-image-edit-with-qwen.git repo_tmp
!cp repo_tmp/server/app_comfyui_gguf.py .
!cp repo_tmp/server/app_comfyui_template.html .
!cp -r repo_tmp/server/comfyui_workflow .
    
print("準備完了\n")
!ls -la app_* comfyui_workflow 2>/dev/null || echo "⚠ ファイルが見つかりません。"

## 6. LoRAのダウンロード（sample）

In [ ]:
%mkdir ./LoRA

#!hf download sbeland/qwen2512-loras --local-dir ./LoRA
#!hf download Kokoboyaw/PanelPainter-Project --local-dir ./hf_download --include "loras/v3/PanelPainter_v3_Qwen2511.safetensors"
#%mv ./hf_download/loras/v3/PanelPainter_v3_Qwen2511.safetensors ./LoRA/

print("OK")

## 7. 設定

パスワードやプリセットを変更できます。

In [ ]:
# --- 設定 ---
PASSWORD = "password"       # 生成パスワード
PORT = 18080                 # サーバーポート
GALLERY = True              # ギャラリーモード有効化

# プリセット (空リストならボタン非表示)
PRESETS = [
    "高画質化::Enhance quality and fix artifacts.",
    "テキスト除去::Remove all overlaid text, subtitles, and credits.",
    "画像1に画像2の服を着せる::将 Image-1 中的角色穿上 Image-2 的服装",
    "LoRA-PanelPainter着彩:Color this panelpainter",
]

COMFYUI_LOG = "./comfyui.log"
SERVER_LOG = "./server.log"

print("OK")

## 8. ComfyUI起動

In [ ]:
import subprocess, time, re, threading

# ログファイルに出力（パイプバッファ詰まり防止）
clog_fh = open(COMFYUI_LOG, "w")

# ComfyUI起動
# --use-flash-attention は使える環境のみ有効にする
comfyui_proc = subprocess.Popen(
    [
        "python", "-u", "./ComfyUI/main.py",
        "--listen",
        "--port", "6006",
        "--preview-method", "auto",
        "--dont-upcast-attention",
#       "--use-flash-attention",
    ],
    stdout=clog_fh, stderr=clog_fh, text=True
)

print("OK")

## 9. サーバー起動

In [ ]:
# コマンドライン引数を構築
cmd = [
    "python", "-u", "./app_comfyui_gguf.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--comfyui-url", "http://127.0.0.1:6006",
    "--comfyui-path", "./ComfyUI",
    "--password", PASSWORD,
    "--steps", "8",
]
if GALLERY:
    cmd.append("--gallery")
for preset in PRESETS:
    cmd.extend(["--preset", preset])

print(f"starting server: {' '.join(cmd)}")

# ログファイルに出力（パイプバッファ詰まり防止）
log_fh = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

# ログを監視してサーバー起動完了を待機
ready = False
deadline = time.time() + 600  # 10分タイムアウト
seen = 0
while time.time() < deadline:
    time.sleep(2)
    if server_proc.poll() is not None:
        print("ERROR: server process exited unexpectedly")
        with open(SERVER_LOG) as f:
            print(f.read())
        break
    with open(SERVER_LOG) as f:
        lines = f.readlines()
    # 新しい行を表示
    for line in lines[seen:]:
        print(line, end="")
    seen = len(lines)
    if any("server starting at" in l for l in lines):
        ready = True
        break

if not ready:
    raise RuntimeError("Server failed to start. Check server.log")

print(f"\n[server is running on port {PORT}")

## 10. cloudflared 公開

cloudflared で公開URLを生成します。
出力に表示される `https://******.trycloudflare.com` のURLにアクセスしてください。

In [ ]:
print(f"starting cloudflared tunnel...")

# cloudflared 起動 (stderr にURL出力)
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

# stderr を読み続けるスレッド（バッファ詰まり防止）
tunnel_lines = []
def drain_tunnel_stderr():
    for line in iter(tunnel_proc.stderr.readline, ""):
        tunnel_lines.append(line)
threading.Thread(target=drain_tunnel_stderr, daemon=True).start()

# URL抽出を待機
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    time.sleep(1)
    for line in tunnel_lines:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            public_url = m.group(1)
            break
    if public_url:
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  Public URL: {public_url}")
    print(f"  Password:   {PASSWORD}")
    print(f"{'='*60}")
    print("30秒ほど待ってからアクセスしてください")
else:
    print("ERROR: cloudflared URL を取得できませんでした")
    print("tunnel log:", ''.join(tunnel_lines[-10:]))

print("バックグラウンドで実行中")

## 11. ユーティリティ

ログ確認・サーバー停止用。

In [ ]:
# サーバーログ確認（直近20行）
!tail -20 ./server.log
print("\n")

# comfyuiログ確認（直近20行）
!tail -20 ./comfyui.log

# サーバー + トンネル停止
```
server_proc.terminate()
log_fh.close()
print("stopped server")

comfyui_proc.terminate()
clog_fh.close()
print("stopped comfyui")

tunnel_proc.terminate()
print("stopped")
```